In [3]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/351.2 MB 6.1 MB/s eta 0:00:58
   ---------------------------------------- 2.1/351.2 MB 5.5 MB/s eta 0:01:03
   ---------------------------------------- 3.7/351.2 MB 6.1 MB/s eta 0:00:57
    --------------------------------------- 5.2/351.2 MB 6.5 MB/s eta 0:00:54
    --------------------------------------- 6.8/351.2 MB 6.8 MB/s eta 0:00:51
    --------------------------------------- 7.9/351.2 MB 7.0 MB/s eta 0:00:50
   - -------------------------------------- 9.4/351.2 MB 6.6 MB/s eta 0:00:53
   - -------------------------------------- 11.0/351.2 MB 6.6 MB/s eta 0:00:52
   - -------------------------------------- 12.6/351.2 MB 6.7 MB/s eta 0:00:51
   - -------------------------------------- 14.2/351.2 MB 6.9 MB/s eta 0:00:50
   - -------------------------------------- 16.3/351.2 MB 7.0 MB/s e

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
# 1. Read dataset
df = pd.read_csv("Churn_Modelling.csv")

In [5]:
# 2. Remove unnecessary columns
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [6]:
# 3. Convert categorical data into numbers
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True)

In [7]:
# 4. Separate features and target
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [8]:
# 5. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
# 6. Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
# 7. Build neural network
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [12]:
# 8. Train the model with early stopping
early_stop = EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7677 - loss: 0.5397 - val_accuracy: 0.7981 - val_loss: 0.4576
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8014 - loss: 0.4479 - val_accuracy: 0.8100 - val_loss: 0.4309
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8075 - loss: 0.4302 - val_accuracy: 0.8131 - val_loss: 0.4221
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8119 - loss: 0.4196 - val_accuracy: 0.8256 - val_loss: 0.4139
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8189 - loss: 0.4104 - val_accuracy: 0.8338 - val_loss: 0.4054
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8238 - loss: 0.4015 - val_accuracy: 0.8400 - val_loss: 0.3956
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8295 - loss: 0.3913 - val_accuracy: 0.8519 - val_loss: 0.3834
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8383 - loss: 0.3804 - val_accuracy: 0.

In [13]:
# 9. Predict customer churn
predictions = (model.predict(X_test) >= 0.5).astype(int).ravel()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [14]:
# 10. Display results
print("\nAccuracy:", accuracy_score(y_test, predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))



Accuracy: 0.8555

Confusion Matrix:
[[1527   66]
 [ 223  184]]
